In [1]:
from pyspark.sql import SparkSession
from delta import *
from pyspark.sql import functions as F

warehouse_location = 'hdfs://hdfs-nn:9000/Projeto/Silver'

builder = SparkSession.builder \
    .appName("Gold - Fact Analise Audio Atomica") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [5]:
# 1. Tabelas Silver (Fontes de Dados)
spark.read.format("delta").load("hdfs://hdfs-nn:9000/warehouse/projeto.db/attributes_and_popularity").createOrReplaceTempView("silver_spotify")
spark.read.format("delta").load("hdfs://hdfs-nn:9000/warehouse/projeto.db/grammy_winners").createOrReplaceTempView("silver_grammys")
spark.read.format("delta").load("hdfs://hdfs-nn:9000/warehouse/projeto.db/soundtracks").createOrReplaceTempView("silver_soundtracks")

# 2. Tabelas Gold (Dimensões já existentes)
spark.read.format("delta").load("hdfs://hdfs-nn:9000/gold/gold_projeto.db/dim_conteudo").createOrReplaceTempView("dim_conteudo")
spark.read.format("delta").load("hdfs://hdfs-nn:9000/gold/gold_projeto.db/dim_tempo").createOrReplaceTempView("dim_tempo")

print("Todas as tabelas (incluindo Tempo) foram carregadas com sucesso.")

Todas as tabelas (incluindo Tempo) foram carregadas com sucesso.


In [6]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Definição da lógica de transformação
df_fact_audio_base = spark.sql("""
    WITH dim_tempo_anual AS (
        SELECT year, FIRST(id_data) as id_tempo 
        FROM dim_tempo 
        GROUP BY year
    ),
    v_grammy_winners AS (
        SELECT DISTINCT LOWER(TRIM(artist)) as artist_clean, 1 as is_winner_flag
        FROM silver_grammys
        WHERE is_winner = true
    ),
    v_spotify_dedup AS (
        SELECT *,
            -- Deduplicação agressiva por nome da música (mantém a mais popular)
            ROW_NUMBER() OVER (
                PARTITION BY LOWER(TRIM(track_name)) 
                ORDER BY CAST(popularity AS DOUBLE) DESC
            ) as rn
        FROM silver_spotify
    ),
    v_soundtracks_dedup AS (
        SELECT DISTINCT 
            LOWER(TRIM(song_name)) as song_name_clean, 
            LOWER(TRIM(name)) as movie_name_clean, 
            year
        FROM silver_soundtracks
    )
    
    SELECT 
        -- 1. Chaves Estrangeiras (FK)
        c.id_conteudo as fk_content,
        t.id_tempo as fk_release_date,
        
        -- 2. Métricas de Áudio (Audio Features)
        CAST(s.popularity AS DOUBLE) as track_popularity_score,
        CAST(s.danceability AS DOUBLE) as danceability_coef,
        CAST(s.energy AS DOUBLE) as energy_coef,
        CAST(s.valence AS DOUBLE) as valence_coef,
        CAST(s.loudness AS DOUBLE) as loudness_coef,
        CAST(s.tempo AS DOUBLE) as tempo_bpm,
        
        -- 3. Flags de Género (Vêm diretamente da Silver como 0 ou 1)
        s.is_rock_metal,
        s.is_pop,
        s.is_hiphop_rap,
        s.is_dance_edm,
        s.is_latin,
        s.is_jazz_blues,
        s.is_classical_ambient,
        s.is_country_folk,
        s.is_reggae,
        s.is_indie,
        s.is_world,
        s.is_soundtrack_children,
        s.is_mood_activity,
        
        -- 4. Atributos Descritivos
        s.track_name,
        s.artists as main_artist,
        s.explicit as is_explicit,
        COALESCE(g.is_winner_flag, 0) as is_grammy_winner

    FROM v_spotify_dedup s
    INNER JOIN v_soundtracks_dedup st ON LOWER(TRIM(s.track_name)) = st.song_name_clean
    INNER JOIN dim_conteudo c ON st.movie_name_clean = LOWER(TRIM(c.title))
    LEFT JOIN dim_tempo_anual t ON st.year = t.year
    LEFT JOIN v_grammy_winners g ON LOWER(TRIM(s.artists)) = g.artist_clean
    
    WHERE s.rn = 1
""")

# --- Adicionar a Primary Key (id_audio_analysis) ---
windowSpec = Window.orderBy("fk_content", "track_name")
df_fact_final = df_fact_audio_base.withColumn("id_audio_analysis", F.row_number().over(windowSpec))

# --- Seleção Final e Ordenação das Colunas ---
# Isto garante que a tabela final tem exatamente as colunas que queremos
cols_final = [
    "id_audio_analysis", "fk_content", "fk_release_date",
    "track_popularity_score", "danceability_coef", "energy_coef", "valence_coef", 
    "loudness_coef", "tempo_bpm",
    "is_explicit",
    "is_rock_metal", "is_pop", "is_hiphop_rap", "is_dance_edm", "is_latin",
    "is_jazz_blues", "is_classical_ambient", "is_country_folk", "is_reggae",
    "is_indie", "is_world", "is_soundtrack_children", "is_mood_activity",
    "track_name", "main_artist", "is_grammy_winner"
]

df_fact_final = df_fact_final.select(*cols_final)

print(f"Volume de dados gerado: {df_fact_final.count()} registos atómicos.")
df_fact_final.printSchema()

Volume de dados gerado: 185 registos atómicos.
root
 |-- id_audio_analysis: integer (nullable = false)
 |-- fk_content: integer (nullable = true)
 |-- fk_release_date: integer (nullable = true)
 |-- track_popularity_score: double (nullable = true)
 |-- danceability_coef: double (nullable = true)
 |-- energy_coef: double (nullable = true)
 |-- valence_coef: double (nullable = true)
 |-- loudness_coef: double (nullable = true)
 |-- tempo_bpm: double (nullable = true)
 |-- is_explicit: boolean (nullable = true)
 |-- is_rock_metal: integer (nullable = true)
 |-- is_pop: integer (nullable = true)
 |-- is_hiphop_rap: integer (nullable = true)
 |-- is_dance_edm: integer (nullable = true)
 |-- is_latin: integer (nullable = true)
 |-- is_jazz_blues: integer (nullable = true)
 |-- is_classical_ambient: integer (nullable = true)
 |-- is_country_folk: integer (nullable = true)
 |-- is_reggae: integer (nullable = true)
 |-- is_indie: integer (nullable = true)
 |-- is_world: integer (nullable = true

In [7]:
target_path = "hdfs://hdfs-nn:9000/gold/gold_projeto.db/fact_analise_audio"

# Remover tabela antiga
spark.sql("DROP TABLE IF EXISTS gold_projeto.fact_analise_audio")

# Gravar nova tabela
df_fact_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", target_path) \
    .saveAsTable("gold_projeto.fact_analise_audio")

print("✅ Tabela 'fact_analise_audio' gravada com sucesso!")

✅ Tabela 'fact_analise_audio' gravada com sucesso!


In [8]:
print("Estrutura Final:")
spark.sql("DESCRIBE TABLE gold_projeto.fact_analise_audio").show()

print("Amostra dos Dados:")
spark.sql("SELECT * FROM gold_projeto.fact_analise_audio LIMIT 5").show()

Estrutura Final:
+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|   id_audio_analysis|      int|   null|
|          fk_content|      int|   null|
|     fk_release_date|      int|   null|
|track_popularity_...|   double|   null|
|   danceability_coef|   double|   null|
|         energy_coef|   double|   null|
|        valence_coef|   double|   null|
|       loudness_coef|   double|   null|
|           tempo_bpm|   double|   null|
|         is_explicit|  boolean|   null|
|       is_rock_metal|      int|   null|
|              is_pop|      int|   null|
|       is_hiphop_rap|      int|   null|
|        is_dance_edm|      int|   null|
|            is_latin|      int|   null|
|       is_jazz_blues|      int|   null|
|is_classical_ambient|      int|   null|
|     is_country_folk|      int|   null|
|           is_reggae|      int|   null|
|            is_indie|      int|   null|
+--------------------+---------+-------+

In [9]:
from pyspark.sql import functions as F

# 1. Carregar as tabelas para validação
df_facto_audio = spark.table("gold_projeto.fact_analise_audio")
df_conteudo = spark.table("gold_projeto.dim_conteudo")

print("--- 1. Visão Geral das Métricas de Áudio ---")
df_facto_audio.select(
    F.count("*").alias("Total_Registos"),
    F.countDistinct("fk_content").alias("Filmes_Unicos"),
    F.countDistinct("track_name").alias("Musicas_Unicas"),
    F.round(F.avg("track_popularity_score"), 2).alias("Media_Popularidade"),
    F.round(F.avg("danceability_coef"), 2).alias("Media_Danceability"),
    F.sum("is_grammy_winner").alias("Total_Vencedores_Grammy")
).show()

print("\n--- 2. Top 10 Músicas Mais Populares (Join com Dim_Conteudo) ---")
# Validamos se os títulos dos filmes e músicas coincidem logicamente
df_facto_audio.join(df_conteudo, df_facto_audio.fk_content == df_conteudo.id_conteudo) \
    .select(
        F.col("title").alias("Titulo_Filme"),
        F.col("track_name").alias("Nome_Musica"),
        F.col("main_artist").alias("Artista"),
        F.round(F.col("track_popularity_score"), 2).alias("Popularidade"),
        F.round(F.col("danceability_coef"), 2).alias("Danceability"),
        F.col("is_grammy_winner").alias("Grammy")
    ) \
    .orderBy(F.desc("Popularidade")) \
    .show(10, truncate=False)

print("\n--- 3. Top 10 Artistas com Mais Músicas em Filmes ---")
df_facto_audio.filter(F.col("main_artist").isNotNull()) \
    .groupBy("main_artist") \
    .agg(
        F.count("*").alias("Qtd_Musicas_Filmes"),
        F.round(F.avg("track_popularity_score"), 2).alias("Media_Popularidade"),
        F.max("is_grammy_winner").alias("Tem_Grammy")
    ) \
    .orderBy(F.desc("Qtd_Musicas_Filmes")) \
    .show(10, truncate=False)

print("\n--- 4. Distribuição de Coeficientes Musicais ---")
df_facto_audio.select(
    F.round(F.avg("danceability_coef"), 3).alias("Media_Danceability"),
    F.round(F.avg("energy_coef"), 3).alias("Media_Energy"),
    F.round(F.avg("valence_coef"), 3).alias("Media_Valence"),
    F.round(F.stddev("danceability_coef"), 3).alias("Desvio_Danceability")
).show()

print("\n--- 5. Verificação de Integridade e Nulos ---")
df_facto_audio.select(
    F.count(F.when(F.col("fk_content").isNull(), 1)).alias("Nulos_Content_FK"),
    F.count(F.when(F.col("fk_release_date").isNull(), 1)).alias("Nulos_Date_FK"),
    F.count(F.when(F.col("track_name").isNull(), 1)).alias("Nulos_Track_Name"),
    F.count(F.when(F.col("main_artist").isNull(), 1)).alias("Nulos_Artist"),
    F.countDistinct("fk_content").alias("Filmes_Unicos"),
    F.countDistinct("fk_content", "track_name").alias("Combinacoes_Unicas")
).show()

print("\n--- 6. Análise de Duplicados ---")
# Agora validamos se o par (fk_content, track_name) é único
duplicados = df_facto_audio.groupBy("fk_content", "track_name", "main_artist").count() \
    .filter(F.col("count") > 1) \
    .orderBy(F.desc("count"))

total_duplicados = duplicados.count()
print(f"Total de combinações duplicadas: {total_duplicados}")

if total_duplicados > 0:
    print("⚠️ ATENÇÃO: Encontrados duplicados!")
    print("\nExemplos de duplicados:")
    duplicados.show(5, truncate=False)
    
    # Mostrar as linhas duplicadas completas
    print("\nDetalhes das linhas duplicadas:")
    duplicados_ids = duplicados.select("fk_content", "track_name").limit(3)
    
    for row in duplicados_ids.collect():
        print(f"\n→ fk_content={row.fk_content}, track_name={row.track_name}")
        df_facto_audio.filter(
            (F.col("fk_content") == row.fk_content) & 
            (F.col("track_name") == row.track_name)
        ).show(truncate=False)
else:
    print("✅ Nenhum duplicado encontrado!")

print("\n--- 7. Validação de Relacionamentos (FKs) ---")
# Verificar se todos os fk_content existem em dim_conteudo
fks_invalidas = df_facto_audio.join(
    df_conteudo, 
    df_facto_audio.fk_content == df_conteudo.id_conteudo, 
    "left_anti"
).count()

print(f"FKs inválidas (fk_content não existe em dim_conteudo): {fks_invalidas}")
if fks_invalidas > 0:
    print("❌ PROBLEMA: Há registos órfãos!")
else:
    print("✅ Todas as FKs são válidas!")

print("\n--- 8. Resumo de Qualidade ---")
total_registos = df_facto_audio.count()
registos_unicos = df_facto_audio.select("fk_content", "track_name").distinct().count()
taxa_duplicacao = round((1 - registos_unicos/total_registos) * 100, 2) if total_registos > 0 else 0

print(f"""
📊 RESUMO FINAL:
   • Total de registos: {total_registos}
   • Combinações únicas (filme+música): {registos_unicos}
   • Taxa de duplicação: {taxa_duplicacao}%
   • Vencedores Grammy: {df_facto_audio.filter(F.col('is_grammy_winner')==1).count()}
   • Filmes com soundtrack: {df_facto_audio.select('fk_content').distinct().count()}
""")

if taxa_duplicacao > 5:
    print("⚠️ ALERTA: Taxa de duplicação superior a 5%!")
elif taxa_duplicacao > 0:
    print("⚠️ Atenção: Existem algumas duplicações.")
else:
    print("✅ Dados limpos sem duplicações!")

--- 1. Visão Geral das Métricas de Áudio ---
+--------------+-------------+--------------+------------------+------------------+-----------------------+
|Total_Registos|Filmes_Unicos|Musicas_Unicas|Media_Popularidade|Media_Danceability|Total_Vencedores_Grammy|
+--------------+-------------+--------------+------------------+------------------+-----------------------+
|           185|           61|           164|             50.63|              0.56|                     20|
+--------------+-------------+--------------+------------------+------------------+-----------------------+


--- 2. Top 10 Músicas Mais Populares (Join com Dim_Conteudo) ---
+--------------------+------------------------+----------------------------+------------+------------+------+
|Titulo_Filme        |Nome_Musica             |Artista                     |Popularidade|Danceability|Grammy|
+--------------------+------------------------+----------------------------+------------+------------+------+
|goodfellas       

In [ ]:
spark.stop()